This file reads RHINO output data (pkl files) and writes them in an openPMD series  
Uses ADIOS2 as a backend (.bp5) with variable-based iteration encoding  
Stores data in particle records (i.e. DataFrame-like)  
Data is stored in a shared folder on NERSC `/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino`

In [ ]:
import openpmd_api as io
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
# Configuration
OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"
DATA_PATH = "/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino/older_data/Data"
SECONDS_PER_DAY = 86400.0

In [ ]:
# Load RHINO Data: time-series and steady-state data
T_ts_df = pd.read_pickle(f"{DATA_PATH}/2025-12-16/06-57-07_AmSC_Generic_FuelCycle_T.pkl")
D_ts_df = pd.read_pickle(f"{DATA_PATH}/2025-12-16/06-57-07_AmSC_Generic_FuelCycle_D.pkl")
T_ss_df = pd.read_pickle(f"{DATA_PATH}/2025-12-16/06-57-07_AmSC_Generic_FuelCycle_T_SteadyState.pkl")
D_ss_df = pd.read_pickle(f"{DATA_PATH}/2025-12-16/06-57-07_AmSC_Generic_FuelCycle_D_SteadyState.pkl")
# Load RHINO Metafile
meta_df = pd.read_pickle(f"{DATA_PATH}/2025-12-16/06-57-07_AmSC_meta.pkl")

In [ ]:
# Convert metadata to dictionary
meta = meta_df[0].to_dict()
for k, v in list(meta.items()):
    if isinstance(v, np.generic):
        meta[k] = v.item()

# Get timestep
dt = float(meta["dt"]) if "dt" in meta else None
if dt is None:
    raise ValueError("dt is required for time-series data")   

In [ ]:
# Define subsystem labels and ids 
labels_subsystem = [
    "Storage_Delivery", 
    "Fueling", 
    "Fusion_Chamber_Pump", 
    "Pd_Cleanup",
    "Protium_Removal", 
    "Exhaust_Processing", 
    "Gas_Detrit", 
    "Water_Detrit",
    "Glovebox", 
    "Stack", 
    "Isotope_Seperation", 
    "Blanket_Extraction",
    "Heat_Exchanger", 
    "Power_Conv_Loop", 
    "Vent_Detrit", 
    "Blanket",
    "Decay_Box", 
    "Stack_Box", 
    "Burn_Box", 
    "Gen_Box", 
    "Uptake_Box"
]
nSubsystems = len(labels_subsystem)
ids_subsystems = np.arange(nSubsystems, dtype=np.uint64)
canon = {name: i for i, name in enumerate(labels_subsystem)}

In [ ]:
# Extract data from pkl files 
# Here we assume that T is defined in all the subsystems and D is defined in some of the subsystems
# Is this a safe assumption?

T_ts = np.ascontiguousarray(T_ts_df.to_numpy(dtype=np.float64))
nSubsystemsT, Nt = T_ts.shape
assert(nSubsystemsT==nSubsystems), "T_ts has the wrong number of subsystems"

D_ts = np.zeros((nSubsystems, Nt), dtype=np.float64)
for name, row in D_ts_df.iterrows():
    D_ts[canon[name], :] = row.to_numpy(dtype=np.float64)

T_ss = T_ss_df.iloc[:, 0].to_numpy(dtype=np.float64)

D_ss = np.zeros((nSubsystems,), dtype=np.float64)
for name, row in D_ss_df.iterrows():
    D_ss[canon[name]] = float(row.iloc[0])

my_inventory = {}
my_inventory["Tritium"]   = {"data_ts": T_ts, "data_ss": T_ss}
my_inventory["Deuterium"] = {"data_ts": D_ts, "data_ss": D_ss}

In [ ]:
# Create Series
adios2_cfg = r'''
{
  "iteration_encoding": "variable_based",
  "adios2": {
    "engine": {
      "type": "bp5",
      "parameters": {
        "StatsLevel": "0"
      }
    }
  }
}
'''
OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"
series = io.Series(OUTPUT_PATH, io.Access_Type.create_linear, adios2_cfg)
print("Writing RHINO data in particle representation...")

In [ ]:
# =================
# Software Metadata
series.set_attribute("software/softwareName", "RHINO")
series.set_attribute("software/softwareDescription", "RHINO: Fusion Pilot Plant fuel cycle simulation")
series.set_attribute("software/softwareVersion", "1.0")
# Placeholder for additional attributes under software/
series.set_attribute("software/versionControlSoftware", "")
series.set_attribute("software/softwareCommit", "")
series.set_attribute("software/softwareDocumentation", "")

In [ ]:
# ===================
# Provenance metadata
series.set_attribute("provenance/author", "Holly Flynn")
series.set_attribute("provenance/authorAffiliation", "Savannah River National Laboratory")
series.set_attribute("provenance/authorEmail", "Holly.Flynn@srnl.doe.gov")
series.set_attribute("provenance/creationDate", "2025-12-16")
# Placeholder for additional attributes under provenance/
series.set_attribute("provenance/inputDirectory", "")
series.set_attribute("provenance/inputFiles", "")
series.set_attribute("provenance/originalDataDirectory", "")
series.set_attribute("provenance/originalDataFiles", "")

In [ ]:
# ==============================
# System metadata (placeholders)
series.set_attribute("system/systemIP", "")
series.set_attribute("system/systemDescription", "")

In [ ]:
# =====================================
# Application metadata (RHINO-specific)
series.set_attribute("metadata/subsystems/description", "Subsystems of the pilot plant fuel cycle")
series.set_attribute("metadata/subsystems/id", ids_subsystems.tolist())
series.set_attribute("metadata/subsystems/labels", labels_subsystem)
# series.set_attribute("metadata/subsystems/mapping", canon)        # dict as an attribute is not supported
# Placeholder for additional attributes under metadata/subsystems/
series.set_attribute("metadata/subsystems/connections", "")
series.set_attribute("metadata/subsystems/connectionType", "")
series.set_attribute("metadata/species/description", "Gas species in the fuel cycle")
series.set_attribute("metadata/species/names", ["Tritium", "Deuterium"])
# Placeholder for additional attributes under metadata/species/
series.set_attribute("metadata/species/Tritium/subsystems", "")
series.set_attribute("metadata/species/Tritium/subsystemsConnection", "")
series.set_attribute("metadata/species/Deuterium/subsystems", "")
series.set_attribute("metadata/species/Deuterium/subsystemsConnection", "")

In [ ]:
series.flush()

In [ ]:
# ====
# Data

series.particles_path = "inventory"

# Subsystems
def write_subsystems(it):
    """
    Write the subsystems data into an inventory record in the OpenPMD series
    """
    subsystems = it.particles["subsystems"]
    subsystems.set_attribute("description", "Subsystems of the pilot plant fuel cycle")

    # Create the record for the subsystem ids
    dataset = io.Dataset(ids_subsystems.dtype, ids_subsystems.shape)
    id_rec = subsystems["id"][io.Record_Component.SCALAR]
    id_rec.reset_dataset(dataset)
    id_rec.store_chunk(ids_subsystems)

In [ ]:
# Variable-based encoding
# Iteration 0: time metadata, subsystems, mass at time step 0
it = series.snapshots()[0]
it.time = 0.0
it.dt = float(dt)
it.time_unit_SI = SECONDS_PER_DAY
it.set_attribute("timeUnitLabel", "day")
write_subsystems(it)
for name in ("Tritium", "Deuterium"):
    s0 = np.ascontiguousarray(my_inventory[name]["data_ts"][:, 0], dtype=np.float64).copy()
    inv = it.particles[name]
    inv.set_attribute("description", "Inventory across subsystems for species " + name)
    inv["mass"].unit_dimension = {io.Unit_Dimension.M: 1}
    rec = inv["mass"][io.Record_Component.SCALAR]
    rec.reset_dataset(io.Dataset(s0.dtype, s0.shape))
    rec.store_chunk(s0)
    rec.unit_SI = 1e-3
it.close()

# Iterations i = time step i (0..Nt-1)
buf_T = np.empty(nSubsystems, dtype=np.float64)
buf_D = np.empty(nSubsystems, dtype=np.float64)
for i in tqdm(range(1, Nt)):
    it = series.write_iterations()[i]
    it.time = float(i) * dt
    it.dt = float(dt)
    it.time_unit_SI = SECONDS_PER_DAY
    it.set_attribute("timeUnitLabel", "day")
    np.copyto(buf_T, my_inventory["Tritium"]["data_ts"][:, i])
    np.copyto(buf_D, my_inventory["Deuterium"]["data_ts"][:, i])
    for name, buf in (("Tritium", buf_T), ("Deuterium", buf_D)):
        inv = it.particles[name]
        inv.set_attribute("description", "Inventory across subsystems for species " + name)
        inv["mass"].unit_dimension = {io.Unit_Dimension.M: 1}
        rec = inv["mass"][io.Record_Component.SCALAR]
        rec.reset_dataset(io.Dataset(buf.dtype, buf.shape))
        rec.store_chunk(buf)
        rec.unit_SI = 1e-3
    it.close()

# Iteration Nt = steady state
it = series.snapshots()[Nt]
it.time = float(Nt) * dt
it.dt = float(dt)
it.time_unit_SI = SECONDS_PER_DAY
it.set_attribute("timeUnitLabel", "day")
for name in ("Tritium", "Deuterium"):
    inv = it.particles[name]
    inv.set_attribute("description", "Inventory across subsystems for species " + name)
    ss = np.ascontiguousarray(my_inventory[name]["data_ss"], dtype=np.float64).copy()
    inv["mass_steady"].unit_dimension = {io.Unit_Dimension.M: 1}
    rec = inv["mass_steady"][io.Record_Component.SCALAR]
    rec.reset_dataset(io.Dataset(ss.dtype, ss.shape))
    rec.store_chunk(ss)
    rec.unit_SI = 1e-3
it.close()

series.close()

In [ ]:
print("RHINO data written to ADIOS-OpenPMD in particle representation.")
print("Output:", OUTPUT_PATH)